In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logomaker
import pyBigWig
from pyfaidx import Fasta

In [ ]:
BASES = ["A", "C", "G", "T"]

def fetch_seq(fa: Fasta, chrom: str, start0: int, end0: int) -> str:
    # pyfaidx uses 0-based slicing, end exclusive
    return str(fa[chrom][start0:end0]).upper()

def fetch_bw_values(bw: pyBigWig.pyBigWig, chrom: str, start0: int, end0: int) -> np.ndarray:
    vals = bw.values(chrom, start0, end0, numpy=True)
    # bw can contain NaN for uncovered positions
    vals = np.array(vals, dtype=float)
    vals[np.isnan(vals)] = 0.0
    return vals

def dynseq_matrix_from_seq_scores(seq: str, scores: np.ndarray) -> pd.DataFrame:
    """
    seq: length L string
    scores: length L float array (can be +/-)
    returns: DataFrame L x 4 (A,C,G,T), where only the observed base gets the score.
    """
    L = len(seq)
    assert scores.shape[0] == L
    mat = np.zeros((L, 4), dtype=float)
    base_to_col = {b:i for i,b in enumerate(BASES)}
    for i, b in enumerate(seq):
        j = base_to_col.get(b, None)
        if j is not None:
            mat[i, j] = scores[i]
        # if N or others -> keep zeros
    return pd.DataFrame(mat, columns=BASES)

def plot_dynseq(df_mat: pd.DataFrame, 
                title: str = "", 
                ylim = (-0.01, 0.03),
                out: str | None = None, 
                dpi: int = 300):
    fig, ax = plt.subplots(figsize=(max(6, df_mat.shape[0] / 10), 2.6))
    logo = logomaker.Logo(df_mat, ax=ax)
    ax.axhline(0, linewidth=1)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_ylim(ylim)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    if out:
        plt.savefig(out, dpi=dpi)
    return fig, ax

In [ ]:
in_dir = '../../results/03_accessbpnet/06_chrombpnet_contribs'
out_dir = '../../results/03_accessbpnet/12_viz_contribution'

os.makedirs(out_dir, exist_ok=True)

In [ ]:

bw_path = "../../results/03_accessbpnet/06_chrombpnet_contribs/K562/ATAC/K562.counts_scores.bw"
fa_path = "../../data/GRCh38/GRCh38.primary_assembly.genome.fa"  # make sure it matches chr naming (chr1 vs 1)

chrom = "chr1"
start0 = 58841500     # 0-based
end0   = 58841800     # end exclusive => length 300bp
fa = Fasta(fa_path)

for assay in ['ACCESS', 'ATAC']:
    bw_path = f"{in_dir}/K562/{assay}/K562.counts_scores.bw"
    bw = pyBigWig.open(bw_path)

    seq = fetch_seq(fa, chrom, start0, end0)
    scores = fetch_bw_values(bw, chrom, start0, end0)
    mat = dynseq_matrix_from_seq_scores(seq, scores)

    plot_dynseq(mat, title="", ylim = (-0.01, 0.1),
                out=f"{out_dir}/{chrom}_{start0}_{end0}_{assay}.pdf")